In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [4]:
!pip install mlflow dagshub -q

In [5]:
import pandas as pd
import numpy as np
import mlflow.pyfunc

In [6]:
import os
import mlflow

os.environ["MLFLOW_TRACKING_USERNAME"] = "lkhar21"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "d2982c921140413d4212c1e34543c62d25a3c42c"

mlflow.set_tracking_uri(
    "https://dagshub.com/lkhar21/ML-assignment2.mlflow"
)

In [7]:
test = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv")

test_ids = test["TransactionID"]
X_test = test.copy()

In [8]:
num_cols = X_test.select_dtypes(include=[np.number]).columns
cat_cols = X_test.select_dtypes(include=["object"]).columns

X_test[num_cols] = X_test[num_cols].fillna(X_test[num_cols].median())
X_test[cat_cols] = X_test[cat_cols].fillna("missing")

# feature engineering
X_test["log_amt"] = np.log1p(X_test["TransactionAmt"])
X_test["null_count"] = X_test.isnull().sum(axis=1)

/tmp/ipykernel_474/1504541197.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_test["log_amt"] = np.log1p(X_test["TransactionAmt"])
/tmp/ipykernel_474/1504541197.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_test["null_count"] = X_test.isnull().sum(axis=1)


In [9]:
full = pd.get_dummies(X_test, columns=cat_cols, drop_first=True)
X_test = full.fillna(0)

In [10]:
import mlflow
import os

os.environ["MLFLOW_TRACKING_USERNAME"] = "lkhar21"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "d2982c921140413d4212c1e34543c62d25a3c42c"

mlflow.set_tracking_uri(
    "https://dagshub.com/lkhar21/ML-assignment2.mlflow"
)

In [11]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

for rm in client.search_registered_models():
    print(rm.name)

Fraud_LogReg_Model
Fraud_LogReg_Model_lkhar21
FRAUD_MODEL
LIGHTGBM_FRAUD


In [12]:
import mlflow.pyfunc

model = mlflow.pyfunc.load_model(
    "models:/LIGHTGBM_FRAUD/1"
)

In [13]:
import pandas as pd
import numpy as np

test = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv")

In [14]:
test = test.fillna(0)

In [15]:
import mlflow.pyfunc

model = mlflow.pyfunc.load_model("models:/LIGHTGBM_FRAUD/1")

In [16]:
import pandas as pd
import numpy as np

test = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv")
test_id = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv")

test = test.merge(test_id, how="left", on="TransactionID")

In [17]:
num_cols = test.select_dtypes(include=[np.number]).columns
cat_cols = test.select_dtypes(include=["object"]).columns

test[num_cols] = test[num_cols].fillna(test[num_cols].median())
test[cat_cols] = test[cat_cols].fillna("missing")

In [18]:
test["log_amount"] = np.log1p(test["TransactionAmt"])
test["null_count"] = test.isnull().sum(axis=1)

/tmp/ipykernel_474/3787628201.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test["log_amount"] = np.log1p(test["TransactionAmt"])
/tmp/ipykernel_474/3787628201.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test["null_count"] = test.isnull().sum(axis=1)


In [19]:
test = pd.get_dummies(test, columns=cat_cols, drop_first=True)

In [20]:
# =========================
# 1. IMPORTS
# =========================
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score

import mlflow
import mlflow.sklearn

In [21]:
df = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv")
df_id = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv")

df = df.merge(df_id, on="TransactionID", how="left")

y = df["isFraud"]
X = df.drop(columns=["isFraud"])

# simple features
X["log_amount"] = np.log1p(X["TransactionAmt"])
X["null_count"] = X.isnull().sum(axis=1)

In [22]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [23]:
cat_cols = X.select_dtypes(include=["object"]).columns
num_cols = X.select_dtypes(exclude=["object"]).columns

preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", "passthrough", num_cols)
    ]
)

In [24]:
model = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=64,
    random_state=42
)

In [25]:
clf = Pipeline(steps=[
    ("prep", preprocess),
    ("model", model)
])

In [26]:
with mlflow.start_run(run_name="LGBM_PIPELINE"):

    clf.fit(X_train, y_train)

    proba = clf.predict_proba(X_val)[:, 1]

    auc = roc_auc_score(y_val, proba)

    print("AUC:", auc)

    mlflow.log_metric("auc", auc)

    mlflow.sklearn.log_model(clf, "model")

[LightGBM] [Info] Number of positive: 16530, number of negative: 455902
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 1.837209 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 40476
[LightGBM] [Info] Number of data points in the train set: 472432, number of used features: 1118
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.034989 -> initscore=-3.317101
[LightGBM] [Info] Start training from score -3.317101


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


AUC: 0.9611952396376099


2026/05/07 20:48:35 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/07 20:48:35 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run LGBM_PIPELINE at: https://dagshub.com/lkhar21/ML-assignment2.mlflow/#/experiments/0/runs/72b51061c6c2486499e320b4d45361c0
🧪 View experiment at: https://dagshub.com/lkhar21/ML-assignment2.mlflow/#/experiments/0


In [27]:
mlflow.sklearn.log_model(
    clf,
    "model",
    registered_model_name="FRAUD_MODEL"
)

2026/05/07 20:48:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/07 20:48:43 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'FRAUD_MODEL' already exists. Creating a new version of this model...
2026/05/07 20:48:52 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: FRAUD_MODEL, version 7
Created version '7' of model 'FRAUD_MODEL'.


In [ ]:
import mlflow.pyfunc

model = mlflow.pyfunc.load_model("models:/FRAUD_MODEL/1")

In [ ]:
!pip install lightgbm mlflow -q

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

from lightgbm import LGBMClassifier
import mlflow
import mlflow.sklearn

In [ ]:
df = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv")
df_id = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv")

df = df.merge(df_id, on="TransactionID", how="left")

y = df["isFraud"]
X = df.drop(columns=["isFraud"])

# missing handling
X = X.fillna(-999)

# simple encoding
X = pd.get_dummies(X, drop_first=True)

In [ ]:
import re

X_train.columns = X_train.columns.astype(str)
X_val.columns = X_val.columns.astype(str)

X_train.columns = [re.sub(r"[^\w]", "_", col) for col in X_train.columns]
X_val.columns = [re.sub(r"[^\w]", "_", col) for col in X_val.columns]

In [ ]:
import mlflow
import mlflow.sklearn

with mlflow.start_run(run_name="LIGHTGBM_FRAUD"):

    model.fit(X_train, y_train)

    # metrics
    proba = model.predict_proba(X_val)[:, 1]

    mlflow.log_metric("roc_auc", roc_auc_score(y_val, proba))

    # SAVE MODEL (IMPORTANT)
    mlflow.sklearn.log_model(
        model,
        artifact_path="model",
        registered_model_name="FRAUD_MODEL"
    )